# Project 3 - RAG tracing

## What we are learning

Answer a question using fictional Cedar Health policies, then follow the evidence through a LangSmith trace. RAG means **retrieval-augmented generation**: find relevant source text and supply it to the model with the question.

```text
Preparation: policies -> sections with IDs -> embeddings -> saved Chroma index
Request: question -> retrieve sections -> format context -> model -> answer
```

By the end, we can locate the retrieved sections, inspect the exact model input, and explain how missing evidence affects an answer.

## Before running

Use the shared `learning-observability/.venv` kernel and `3_rag_tracing` as the working directory. Keep API keys in the repository root `.env`, with `LANGSMITH_TRACING=true`.

Prepare the index separately from a terminal in this numbered folder:

```powershell
uv run python -m tools.prepare_data
```

The expected preparation output is `Policy sections indexed: 6`. Each policy section is one retrieval unit. Questions and draft reference answers are kept separately from the indexed policy documents.

Run the notebook cells in order. After editing an imported helper, restart the kernel and rerun the setup cells. The model-calling sections each send a request when executed. See [readme.md](readme.md) for the files, data, and preparation details.

## 1. Load the index and retrieve evidence

Load the root environment settings and open the existing Chroma collection. `load_vector_store()` uses the same embedding model and collection settings as preparation. It opens the saved index for reuse.

`retrieve_sections()` searches for the three closest policy sections because `k=3`. Each result has `page_content` for its text and `metadata` for its section ID and filename. These identifiers let us follow a fact back to its source.

For the annual-leave question, look for `LEAVE-001`, which states 18 days per calendar year. Read the returned text to establish relevance; similarity alone does not establish that a section answers the question.

In [1]:
from dotenv import load_dotenv
from models.vector_store import PROJECT_ROOT, load_vector_store

load_dotenv(PROJECT_ROOT.parent / ".env")

from tools.rag import retrieve_sections

vector_store = load_vector_store()

question = "How many days of paid annual leave do employees receive?"
documents = retrieve_sections(question, vector_store, k=3)

for document in documents:
    print(document.metadata["section_id"], document.metadata["source"])
    print(document.page_content, "\n")

<repo>\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3057.82it/s]


LEAVE-001 leave_policy.md
LEAVE-001: Annual leave entitlement

Full-time employees receive 18 days of paid annual leave per calendar year. 

LEAVE-002 leave_policy.md
LEAVE-002: Requesting annual leave

Employees must submit annual leave requests at least five working days
before their planned leave begins.

The employee's reporting manager must approve the request before
the leave is confirmed. 

NOTICE-001 notice_policy.md
NOTICE-001: Notice period

Full-time employees must serve a notice period of 30 calendar days
when resigning. 



<repo>\.venv\Lib\site-packages\langsmith\_internal\_addressing.py:104: LangSmithBetaWarning: Addressing runs with `langsmith.address` is in beta and is enabled per workspace. A workspace without it rejects the run, so the trace is lost rather than falling back to a project. The behavior may change without notice.
  _warn_once(


## 2. Turn the retrieved documents into context

`format_context()` combines the retrieved documents into one string. Each block includes a section ID in square brackets, its filename, and its policy text.

Inspect this output before generating an answer: this is the evidence that will be inserted into the prompt. The trace of `format_context` connects the document inputs to this text output.

In [2]:
from tools.rag import format_context

context = format_context(documents)
print(context)

[LEAVE-001] leave_policy.md
LEAVE-001: Annual leave entitlement

Full-time employees receive 18 days of paid annual leave per calendar year.

[LEAVE-002] leave_policy.md
LEAVE-002: Requesting annual leave

Employees must submit annual leave requests at least five working days
before their planned leave begins.

The employee's reporting manager must approve the request before
the leave is confirmed.

[NOTICE-001] notice_policy.md
NOTICE-001: Notice period

Full-time employees must serve a notice period of 30 calendar days
when resigning.


## 3. Inspect the complete prompt

Read the instructions from `prompts/rag_answer.txt` and fill its `{context}` and `{question}` placeholders.

The instructions ask for answers grounded in the supplied policies, citations using their section IDs, and a clear acknowledgment when information is missing. The printed prompt makes the question, evidence, and instructions visible together.

Draft reference answers are used separately for human inspection; they are not inserted into this prompt.

In [3]:
prompt_template = (
    PROJECT_ROOT / "prompts" / "rag_answer.txt"
).read_text(encoding="utf-8")

prompt = prompt_template.format(context=context, question=question)
print(prompt)

You are an HR policy assistant for Cedar Health, a fictional company.

Answer the question using the policy context below.

Instructions:
- Use only facts stated in the supplied context.
- Cite the supporting section ID after each policy claim, using square brackets.
- Use only section IDs present in the context.
- If information is missing, clearly say that the supplied policies do not specify it.
- For a question with multiple parts, answer the supported parts and identify any missing information.
- Keep the answer short and clear.

Policy context:
[LEAVE-001] leave_policy.md
LEAVE-001: Annual leave entitlement

Full-time employees receive 18 days of paid annual leave per calendar year.

[LEAVE-002] leave_policy.md
LEAVE-002: Requesting annual leave

Employees must submit annual leave requests at least five working days
before their planned leave begins.

The employee's reporting manager must approve the request before
the leave is confirmed.

[NOTICE-001] notice_policy.md
NOTICE-001

## 4. Generate one answer directly

`load_llm()` creates the Groq model client using the settings in `models/llm.py`. The client receives the prompt assembled above. `response.content` is the answer text.

The tracing context routes this call to **rag-tracing -> Local**. LangChain records the model call automatically. Open that run in LangSmith to inspect the actual prompt and response.

Expected behavior for this example: an answer giving 18 days of paid annual leave, supported by `[LEAVE-001]`. This standalone call teaches the generation step before we connect the whole application.

In [4]:
from langsmith import agent_address, tracing_context
from models.llm import load_llm

llm = load_llm()

with tracing_context(address=agent_address("rag-tracing", "local")):
    response = llm.invoke(prompt)

print(response.content)

Employees receive 18 days of paid annual leave per calendar year. [LEAVE-001]


## 5. Follow one complete RAG request

`answer_question()` connects the existing helpers and the model call. Its decorator records a parent run; the nested operations appear as child runs:

```text
answer_question
|-- retrieve_sections
|-- format_context
`-- ChatGroq
```

In **rag-tracing -> Local -> Tracing**, expand the latest `answer_question` run. Follow the document IDs from retrieval, through context formatting, into the model input and final answer.

The parent duration measures the request. Index preparation and opening the vector store happen separately, before this request. This cell makes another model call as part of the complete flow.

In [5]:
from models.llm import load_llm
from tools.rag import answer_question

llm = load_llm()

answer = answer_question(
    question=question,
    vector_store=vector_store,
    llm=llm,
    prompt_template=prompt_template,
    k=3,
)

print(answer)

Employees receive 18 days of paid annual leave per calendar year. [LEAVE-001]


## 6. Ask a question the policies cannot answer

The supplied policies specify annual leave, while this question asks about paid sick leave. Search can still return similar sections even when they do not establish the requested fact.

Expected behavior: the answer acknowledges that the supplied policies do not specify a paid sick-leave entitlement. Inspect retrieval, formatted context, and the model input to understand the evidence available.

An honest acknowledgment of missing information is a successful result here. A run's success status describes execution; answer quality still needs inspection against the source.

In [6]:
unsupported_question = (
    "How many days of paid sick leave do full-time employees "
    "receive per calendar year?"
)

unsupported_answer = answer_question(
    question=unsupported_question,
    vector_store=vector_store,
    llm=llm,
    prompt_template=prompt_template,
    k=3,
)

print(unsupported_answer)

The supplied policies do not specify the number of paid sick leave days for full‑time employees.


## 7. Remove context for a question with a known answer

This controlled diagnostic reuses the original annual-leave `question` and the same model and prompt instructions, while supplying `context=""`. The `answer` variable holds the earlier complete-RAG answer.

The diagnostic calls the model directly. Its trace is named `missing_context_demo`; inspect its model input to see the empty policy-context block. Compare it with the model input inside the earlier complete RAG trace.

Expected behavior: the supported answer gives 18 days with a source citation, while the empty-context answer reports missing policy information. A specific entitlement in the empty-context answer would be unsupported by the supplied evidence.

Here the source fact exists in the dataset, but we deliberately omit it from the model input. That is a different cause from the sick-leave example, where the source policies themselves lack the requested fact. Compare answer behavior in this exercise; a complete-request duration and a model-only duration cover different operations.

In [8]:
from langsmith import agent_address, tracing_context

missing_context_prompt = prompt_template.format(
    context="",
    question=question,
)

with tracing_context(address=agent_address("rag-tracing", "local")):
    missing_context_response = llm.invoke(
        missing_context_prompt,
        config={"run_name": "missing_context_demo"},
    )

print("Question:", question)
print("\nWith context:", answer)
print("\nWithout context:", missing_context_response.content)

Question: How many days of paid annual leave do employees receive?

With context: Employees receive 18 days of paid annual leave per calendar year. [LEAVE-001]

Without context: The supplied policies do not specify how many days of paid annual leave employees receive.


## Saved observations from this notebook

These notes summarize the outputs already saved in the notebook before this documentation update. The model calls were not rerun while adding the Markdown.

| Example | Saved output | What it demonstrates |
| --- | --- | --- |
| Complete RAG request for annual leave | 18 days of paid annual leave per calendar year, citing `[LEAVE-001]` | The answer connects to a fact in the supplied policy |
| Question about paid sick leave | The supplied policies do not specify the number of days | The application acknowledges an information gap in the source documents |
| Annual-leave question with empty context | The supplied policies do not specify the annual-leave entitlement | Removing the supplied evidence changes the response to the same question |

The learner also confirmed that the new lesson's traces are visible in LangSmith. Trace links and exported reports have not been added to this notebook. The six-case JSON question set remains available for later use; its reference answers still have draft review status.

## What we learned

1. **Preparation and requests have different jobs.** Preparation builds the index; questions reuse it. Changing retrieval `k` alone does not require re-indexing.
2. **Embeddings make similarity search possible.** Similar sections can be returned even when they lack the fact needed to answer a question.
3. **Source IDs connect the evidence.** A section such as `LEAVE-001` can be followed through retrieval, formatted context, the model prompt, and the answer citation.
4. **The model receives the assembled prompt.** Inspecting the actual model input helps explain a result even when the source document on disk contains the right fact.
5. **Decorators make function execution visible.** `@traceable` records inputs, outputs, duration, and status. Calls between decorated functions create the parent/child structure.
6. **A successful run still needs a quality check.** Read the supporting policy text to establish whether an answer is grounded. A correct acknowledgment of missing information can be a successful answer.
7. **Missing knowledge and missing context are different problems.** The sick-leave fact is absent from the policies; the annual-leave fact exists but was intentionally omitted in the diagnostic.
8. **Trace destinations and timing boundaries matter.** Use the `rag-tracing` agent's Local environment. Compare durations only when they cover the same operation.

The core learning loop is present: inspect a supported answer, explore an unsupported question, change one input condition, and compare the same question with and without evidence.

## Where we are now

The core Project 3 flow and its saved examples are documented. The saved outputs demonstrate these particular cases; the broader reference set still needs human review before use as trusted goldens. Start another project only when the learner chooses to continue.